In [ ]:
"""
Wong & Wang (2006) reduced two-variable model -- Appendix version (no AMPA at recurrent synapses).

    dS_i/dt = -S_i/tau_S + (1 - S_i) * gamma * H(x_i)
    H(x)    = (a*x - b) / (1 - exp(-d*(a*x - b)))
    x_1     = J_N11*S_1 - J_N12*S_2 + I0 + J_Aext*mu0*(1 + c) + I_noise_1
    x_2     = J_N22*S_2 - J_N21*S_1 + I0 + J_Aext*mu0*(1 - c) + I_noise_2
    tau_AMPA * dI_noise_i/dt = -I_noise_i + eta_i * sqrt(tau_AMPA) * sigma

c is the coherence as a FRACTION (0.064 = 6.4 %).  mu0 in Hz.  Time in seconds, currents in nA.
Cross-coupling is stored positive (JN12 = 0.0497) and subtracted, as in the paper's Appendix.
"""
import numpy as np
from scipy.optimize import brentq

P = dict(
    a=270.0, b=108.0, d=0.154,          # H(x): Hz/nA, Hz, s
    gamma=0.641, tau_S=0.100,           # NMDA gating
    JN11=0.2609, JN12=0.0497,           # nA  (JN22 = JN11, JN21 = JN12)
    I0=0.3255, JAext=5.2e-4,            # nA, nA/Hz
    sigma=0.02, tau_noise=0.002,        # nA, s   (main-text value is 0.007 nA, with AMPA)
    theta=15.0,                         # decision threshold, Hz
)

def H(x, p=P):
    z = p["a"]*x - p["b"]
    z = np.where(np.abs(z) < 1e-9, 1e-9, z)          # removable singularity at z = 0 (limit = 1/d)
    return z/(1.0 - np.exp(-p["d"]*z))

def currents(S1, S2, mu0, c, p=P, wscale=1.0):
    """Noise-free input currents; wscale multiplies the self-excitation JN11 (crude proxy for w+)."""
    x1 = wscale*p["JN11"]*S1 - p["JN12"]*S2 + p["I0"] + p["JAext"]*mu0*(1+c)
    x2 = wscale*p["JN11"]*S2 - p["JN12"]*S1 + p["I0"] + p["JAext"]*mu0*(1-c)
    return x1, x2

def rhs(S1, S2, mu0, c, p=P, wscale=1.0):
    x1, x2 = currents(S1, S2, mu0, c, p, wscale)
    f = lambda S, x: -S/p["tau_S"] + (1-S)*p["gamma"]*H(x, p)
    return f(S1, x1), f(S2, x2)

# ---------------------------------------------------------------- steady states
def nullcline_S2(S1, mu0, c, p=P, wscale=1.0, n=4000):
    """Points (S1, S2) on dS1/dt = 0. Solved for S1 given S2 (explicit for S2->S1 is not closed-form),
    so we scan S2 on a grid and root-find S1."""
    S2 = np.linspace(1e-4, 0.999, n)
    out = []
    for s2 in S2:
        g = lambda s1: rhs(s1, s2, mu0, c, p, wscale)[0]
        if g(1e-6)*g(0.9999) < 0:
            out.append((brentq(g, 1e-6, 0.9999), s2))
    return np.array(out)

def fixed_points(mu0, c=0.0, p=P, wscale=1.0, n=60):
    """All fixed points via many Newton-like starts on a grid; returns list of (S1, S2, stable)."""
    from scipy.optimize import fsolve
    G = lambda S: np.array(rhs(S[0], S[1], mu0, c, p, wscale))
    pts = []
    for s1 in np.linspace(0.005, 0.95, 12):
        for s2 in np.linspace(0.005, 0.95, 12):
            sol, _, ier, _ = fsolve(G, [s1, s2], full_output=True, xtol=1e-12)
            if ier == 1 and np.all((sol > 0) & (sol < 1)) and np.max(np.abs(G(sol))) < 1e-8:
                if not any(np.allclose(sol, q[:2], atol=1e-5) for q in pts):
                    h = 1e-6
                    J = np.column_stack([(G(sol+[h, 0]) - G(sol-[h, 0]))/(2*h),
                                         (G(sol+[0, h]) - G(sol-[0, h]))/(2*h)])
                    pts.append((sol[0], sol[1], bool(np.all(np.linalg.eigvals(J).real < 0))))
    return pts

def spontaneous_state(p=P, wscale=1.0, T=3.0, dt=1e-3):
    S = np.array([0.1, 0.1])
    for _ in range(int(T/dt)):
        d = rhs(S[0], S[1], 0.0, 0.0, p, wscale)
        S = S + dt*np.array(d)
    return S

# ---------------------------------------------------------------- stochastic simulation
def simulate(mu0, c=0.0, ntrials=1, T=3.0, dt=5e-4, p=P, wscale=1.0, S0=None, seed=None,
             stop_at_threshold=True):
    """Euler-Maruyama, vectorised over trials. Returns dict with t, S (T,2,n), r (T,2,n), RT (n), choice (n).
    choice = 0 -> population 1 crossed theta first, 1 -> population 2, -1 -> none within T."""
    rng = np.random.default_rng(seed)
    nT = int(T/dt)
    S0 = spontaneous_state(p, wscale) if S0 is None else np.asarray(S0, float)
    S = np.tile(S0[:, None], (1, ntrials))
    n = np.zeros((2, ntrials))
    Sh = np.empty((nT, 2, ntrials)); rh = np.empty_like(Sh)
    RT = np.full(ntrials, np.nan); choice = np.full(ntrials, -1)
    alive = np.ones(ntrials, bool)
    k_noise = p["sigma"]*np.sqrt(dt/p["tau_noise"])
    for k in range(nT):
        n += -dt/p["tau_noise"]*n + k_noise*rng.standard_normal(n.shape)      # OU noise
        x1, x2 = currents(S[0], S[1], mu0, c, p, wscale)
        r = np.vstack([H(x1 + n[0], p), H(x2 + n[1], p)])
        S = S + dt*(-S/p["tau_S"] + (1-S)*p["gamma"]*r)
        Sh[k], rh[k] = S, r
        hit = alive & (r.max(axis=0) > p["theta"])
        RT[hit] = (k+1)*dt; choice[hit] = r[:, hit].argmax(axis=0); alive &= ~hit
        if stop_at_threshold and not alive.any():
            Sh, rh = Sh[:k+1], rh[:k+1]; break
    return dict(t=np.arange(Sh.shape[0])*dt, S=Sh, r=rh, RT=RT, choice=choice)

def psychometric(cs=(0, .032, .064, .128, .256, .512), mu0=30.0, ntrials=500, **kw):
    out = []
    for c in cs:
        res = simulate(mu0, c, ntrials=ntrials, **kw)
        ok = res["choice"] >= 0
        out.append((c, np.mean(res["choice"][ok] == 0) if c >= 0 else np.nan,
                    np.nanmean(res["RT"]), ok.mean()))
    return out        # (coherence, P(choose pop 1), mean RT [s], fraction that decided)

# ---------------------------------------------------------------- quick look
if __name__ == "__main__":
    import matplotlib.pyplot as plt
    print("spontaneous state S:", spontaneous_state(), " rate (Hz):", H(np.array(currents(*spontaneous_state(), 0, 0))))
    for mu0 in (0.0, 30.0):
        print(f"mu0={mu0}:", [(round(a, 3), round(b, 3), 'stable' if s else 'unstable') for a, b, s in fixed_points(mu0)])
    res = simulate(30.0, 0.064, ntrials=5, seed=1)
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
    ax[0].plot(res["t"], res["r"][:, 0, :], "b", lw=.8); ax[0].plot(res["t"], res["r"][:, 1, :], "r", lw=.8)
    ax[0].axhline(P["theta"], color="k", ls=":"); ax[0].set_xlabel("t (s)"); ax[0].set_ylabel("rate (Hz)")
    for c_ in (0.0, 0.064):
        nc = nullcline_S2(None, 30.0, c_)
        ax[1].plot(nc[:, 0], nc[:, 1], label=f"dS1/dt=0, c={c_}")
        ax[1].plot(nc[:, 1], nc[:, 0], "--", label=f"dS2/dt=0, c={c_}")
    ax[1].set_xlabel("S1"); ax[1].set_ylabel("S2"); ax[1].legend(fontsize=6)
    plt.tight_layout(); plt.show()